In [6]:
import math

# BOARD UTILITIES

def create_board():
    return [' '] * 9

def print_board(board, move_number=None):
    if move_number is not None:
        print("\nBoard after Move", move_number)
    for row in range(3):
        cells = []
        for col in range(3):
            idx = row * 3 + col
            cell = board[idx] if board[idx] != ' ' else str(idx + 1)
            cells.append(" " + cell + " ")
        print("|".join(cells))
        if row < 2:
            print("-----------")

def get_winner(board):
    wins = [
        (0,1,2), (3,4,5), (6,7,8),
        (0,3,6), (1,4,7), (2,5,8),
        (0,4,8), (2,4,6)
    ]
    for a, b, c in wins:
        if board[a] == board[b] == board[c] and board[a] != ' ':
            return board[a]
    return None

def is_full(board):
    return ' ' not in board

def is_terminal(board):
    return get_winner(board) is not None or is_full(board)

def get_utility(board):
    winner = get_winner(board)
    if winner == 'X':
        return 1
    elif winner == 'O':
        return -1
    return 0

def get_available_moves(board):
    return [i for i, cell in enumerate(board) if cell == ' ']


# ALPHA-BETA MINIMAX

stats = {"cutoffs": 0, "nodes_visited": 0}

def minimax_alpha_beta(board, depth, is_maximizing, alpha, beta, verbose=False, path="root"):
    stats["nodes_visited"] += 1

    if is_terminal(board):
        util = get_utility(board)
        if verbose:
            print("TERMINAL depth=", depth, "util=", util, "path=", path)
        return util, None

    best_move = None

    if is_maximizing:
        best_val = -math.inf
        player = 'X'
        for move in get_available_moves(board):
            board[move] = player
            val, _ = minimax_alpha_beta(board, depth + 1, False, alpha, beta,
                                        verbose, path + "->X" + str(move+1))
            board[move] = ' '

            if verbose:
                print("MAX move=", move+1, "val=", val, "alpha=", alpha, "beta=", beta)

            if val > best_val:
                best_val = val
                best_move = move

            alpha = max(alpha, best_val)

            if best_val >= beta:
                stats["cutoffs"] += 1
                if verbose:
                    print("BETA CUTOFF at move", move+1)
                break

        return best_val, best_move

    else:
        best_val = math.inf
        player = 'O'
        for move in get_available_moves(board):
            board[move] = player
            val, _ = minimax_alpha_beta(board, depth + 1, True, alpha, beta,
                                        verbose, path + "->O" + str(move+1))
            board[move] = ' '

            if verbose:
                print("MIN move=", move+1, "val=", val, "alpha=", alpha, "beta=", beta)

            if val < best_val:
                best_val = val
                best_move = move

            beta = min(beta, best_val)

            if best_val <= alpha:
                stats["cutoffs"] += 1
                if verbose:
                    print("ALPHA CUTOFF at move", move+1)
                break

        return best_val, best_move


# AGENT CLASS

class Agent:
    def __init__(self, name, symbol, is_max):
        self.name = name
        self.symbol = symbol
        self.is_max = is_max
        self.total_utility = 0
        self.move_count = 0

    def choose_move(self, board, verbose_search=True):
        print("\n", self.name, "(", self.symbol, ") is thinking...")

        val, move = minimax_alpha_beta(
            board,
            depth=0,
            is_maximizing=self.is_max,
            alpha=-math.inf,
            beta=math.inf,
            verbose=verbose_search
        )

        self.move_count += 1
        self.total_utility += val

        print(self.name, "chooses cell", move+1)
        print("Utility:", val)
        print("Total Utility:", self.total_utility)

        return move


# GAME SIMULATION

def simulate_game(verbose_search=True):
    board = create_board()
    MAX_agent = Agent("MAX", 'X', True)
    MIN_agent = Agent("MIN", 'O', False)

    stats["cutoffs"] = 0
    stats["nodes_visited"] = 0

    print("\nTIC-TAC-TOE: MAX (X) vs MIN (O)")
    print("Using Minimax with Alpha-Beta Pruning\n")

    print_board(board)

    agents = [MAX_agent, MIN_agent]
    move_number = 0

    while not is_terminal(board):
        current_agent = agents[move_number % 2]

        move = current_agent.choose_move(board, verbose_search)
        board[move] = current_agent.symbol
        move_number += 1

        print_board(board, move_number)

    winner = get_winner(board)

    print("\nFinal Result:")
    if winner == 'X':
        print("MAX (X) WINS")
    elif winner == 'O':
        print("MIN (O) WINS")
    else:
        print("DRAW")

    print("\nFinal Utilities:")
    print("MAX:", MAX_agent.total_utility)
    print("MIN:", MIN_agent.total_utility)

    print("\nSearch Stats:")
    print("Nodes visited:", stats["nodes_visited"])
    print("Cutoffs:", stats["cutoffs"])


# MAIN

if __name__ == "__main__":
    verbose= False #by default i have made it false because showing alpha beta cut offs after every move gives a very big output, 
    #if we make verbose=True the output will show the alpha beta values as well
    simulate_game(verbose_search=verbose)


TIC-TAC-TOE: MAX (X) vs MIN (O)
Using Minimax with Alpha-Beta Pruning

 1 | 2 | 3 
-----------
 4 | 5 | 6 
-----------
 7 | 8 | 9 

 MAX ( X ) is thinking...
MAX chooses cell 1
Utility: 0
Total Utility: 0

Board after Move 1
 X | 2 | 3 
-----------
 4 | 5 | 6 
-----------
 7 | 8 | 9 

 MIN ( O ) is thinking...
MIN chooses cell 5
Utility: 0
Total Utility: 0

Board after Move 2
 X | 2 | 3 
-----------
 4 | O | 6 
-----------
 7 | 8 | 9 

 MAX ( X ) is thinking...
MAX chooses cell 2
Utility: 0
Total Utility: 0

Board after Move 3
 X | X | 3 
-----------
 4 | O | 6 
-----------
 7 | 8 | 9 

 MIN ( O ) is thinking...
MIN chooses cell 3
Utility: 0
Total Utility: 0

Board after Move 4
 X | X | O 
-----------
 4 | O | 6 
-----------
 7 | 8 | 9 

 MAX ( X ) is thinking...
MAX chooses cell 7
Utility: 0
Total Utility: 0

Board after Move 5
 X | X | O 
-----------
 4 | O | 6 
-----------
 X | 8 | 9 

 MIN ( O ) is thinking...
MIN chooses cell 4
Utility: 0
Total Utility: 0

Board after Move 6
 X |